In [23]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from xgboost import XGBRegressor

In [24]:
train = pd.read_csv("data/dataset/train.csv")
test = pd.read_csv("data/dataset/test.csv")

print(train.shape, test.shape)

(77299, 11) (41778, 10)


In [25]:
TARGET = "demand"

X = train.drop(columns=[TARGET])
y = train[TARGET]

In [34]:
for df in [X, test]:

    # time
    df["hour"] = pd.to_datetime(df["timestamp"], errors="coerce").dt.hour
    df["minute"] = pd.to_datetime(df["timestamp"], errors="coerce").dt.minute

    df["time_minutes"] = df["hour"] * 60 + df["minute"]

    # cyclical encoding (VERY IMPORTANT)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    # demand pattern proxies
    df["is_rush_hour"] = df["hour"].isin([8,9,17,18,19]).astype(int)
    df["is_night"] = (df["hour"] <= 5).astype(int)

    df["is_weekend_like"] = (df["day"] % 7).isin([5,6]).astype(int)

KeyError: 'timestamp'

In [35]:
def freq_encode(train_df, test_df, col):
    freq = train_df[col].value_counts(normalize=True)
    train_df[col + "_freq"] = train_df[col].map(freq)
    test_df[col + "_freq"] = test_df[col].map(freq)
    return train_df, test_df


for col in ["geohash", "RoadType", "Weather", "LargeVehicles", "Landmarks"]:
    X, test = freq_encode(X, test, col)

KeyError: 'geohash'

In [28]:
drop_cols = ["geohash", "RoadType", "Weather", "LargeVehicles", "Landmarks"]

X = X.drop(columns=drop_cols)
test_processed = test.drop(columns=drop_cols)

In [33]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for train_idx, val_idx in kf.split(X):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_tr, y_tr)

    preds = model.predict(X_val)

    score = r2_score(y_val, preds)
    scores.append(score)

print("FINAL CV R2:", np.mean(scores))

FINAL CV R2: 0.8048924951051172
